# W03 — FAISS RAG Pipeline (Fari eldercare / Senpai grounding)

Reuses the architecture pattern from the Psychologist Agent's CBT/DBT/WHO-grounded RAG pipeline (sentence-transformers embeddings + FAISS) and, on the normalization side, OpenClaw's Knowledge agent (384-dim L2-normalized embeddings). Knowledge base: `knowledge_base.json`, 15 original chunks summarizing well-established eldercare-communication and person-centered-care principles, written in my own words rather than copied from a specific CDC/WHO document, to keep this repo free of any copyright ambiguity while still being grounded in real practice.

Steps: embedding generation → FAISS indexing → retrieval evaluation on a held-out query set (`eval_queries.json`, written independently of the pipeline build) → top-k hit-rate report.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer

NOTEBOOK_DIR = Path.cwd()
SEED = 42
np.random.seed(SEED)

EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"  # 384-dim, matches OpenClaw's embedding size

/Users/yihangsun/Desktop/2026找工/8 bit/inGenInternshipDataSources/yihang-ingen-ml-nn-intern/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load knowledge base

In [2]:
with open(NOTEBOOK_DIR / "knowledge_base.json") as f:
    kb = json.load(f)

chunks = kb["chunks"]
chunk_ids = [c["chunk_id"] for c in chunks]
chunk_texts = [c["text"] for c in chunks]
print(f"Loaded {len(chunks)} knowledge base chunks")
pd.DataFrame(chunks)[["chunk_id", "topic"]]

Loaded 15 knowledge base chunks


,chunk_id,topic
0,validation_listening,Active listening and emotional validation
1,loneliness_response,Responding to expressed loneliness or isolation
2,grief_support,Grief support and anniversary awareness
3,dignity_autonomy,Preserving dignity and autonomy while assisting
4,medication_reminders,Person-centered medication reminders
5,disorientation_dementia,Communicating with someone who is disoriented ...
6,elderspeak_avoidance,Avoiding patronizing tone ('elderspeak')
7,pain_acknowledgment,Acknowledging reported pain or discomfort
8,family_connection,Facilitating family connection
9,anxiety_appointments,Supporting pre-appointment anxiety


## 2. Embedding generation

In [3]:
embedder = SentenceTransformer(EMBED_MODEL_NAME)

chunk_embeddings = embedder.encode(chunk_texts, convert_to_numpy=True, show_progress_bar=False)
chunk_embeddings = chunk_embeddings.astype("float32")

# L2-normalize so inner product == cosine similarity (same pattern OpenClaw's Knowledge agent uses)
faiss.normalize_L2(chunk_embeddings)

print(f"Embedding shape: {chunk_embeddings.shape}")

Embedding shape: (15, 384)


## 3. FAISS indexing

In [4]:
embedding_dim = chunk_embeddings.shape[1]
index = faiss.IndexFlatIP(embedding_dim)  # inner product on normalized vectors = cosine similarity
index.add(chunk_embeddings)

print(f"FAISS index built: {index.ntotal} vectors, dim={embedding_dim}")


def retrieve(query, top_k=5):
    q_emb = embedder.encode([query], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(q_emb)
    scores, indices = index.search(q_emb, top_k)
    return [
        {"chunk_id": chunk_ids[i], "topic": chunks[i]["topic"], "score": float(s)}
        for s, i in zip(scores[0], indices[0])
    ]


# quick sanity check on a query not in the held-out eval set
for r in retrieve("a resident says they don't feel like waking up tomorrow", top_k=3):
    print(f"{r['score']:.3f}  {r['chunk_id']:<25}  {r['topic']}")

FAISS index built: 15 vectors, dim=384


0.394  disorientation_dementia    Communicating with someone who is disoriented or confused
0.382  anxiety_appointments       Supporting pre-appointment anxiety
0.375  crisis_recognition         General signs a conversation may involve elevated risk


## 4. Retrieval evaluation (held-out query set)

`eval_queries.json` was written after the knowledge base and pipeline were already built, with each query phrased differently from its target chunk's own wording (not a near-restatement of the chunk text), and it isn't used anywhere in building the index or tuning retrieval -- only here, for evaluation.

In [5]:
with open(NOTEBOOK_DIR / "eval_queries.json") as f:
    eval_data = json.load(f)

eval_queries = eval_data["queries"]
print(f"Loaded {len(eval_queries)} held-out evaluation queries")


def hit_at_k(results, relevant_id, k):
    return relevant_id in [r["chunk_id"] for r in results[:k]]


K_VALUES = [1, 3, 5]
rows = []
for q in eval_queries:
    results = retrieve(q["query"], top_k=max(K_VALUES))
    row = {"query": q["query"], "relevant_chunk_id": q["relevant_chunk_id"]}
    for k in K_VALUES:
        row[f"hit@{k}"] = hit_at_k(results, q["relevant_chunk_id"], k)
    row["top1_retrieved"] = results[0]["chunk_id"]
    rows.append(row)

eval_df = pd.DataFrame(rows)
eval_df

Loaded 15 held-out evaluation queries


,query,relevant_chunk_id,hit@1,hit@3,hit@5,top1_retrieved
0,My resident hasn't heard from her kids in a wh...,loneliness_response,False,True,True,family_connection
1,How should staff acknowledge a resident's ongo...,pain_acknowledgment,True,True,True,pain_acknowledgment
2,It's the anniversary of my resident's husband ...,grief_support,True,True,True,grief_support
3,A resident gets annoyed and refuses to keep co...,medication_reminders,True,True,True,medication_reminders
4,Someone suddenly seems unsure where they are o...,disorientation_dementia,True,True,True,disorientation_dementia
5,Why is it a problem to talk to older adults in...,elderspeak_avoidance,True,True,True,elderspeak_avoidance
6,How do I help someone feel like they're still ...,dignity_autonomy,True,True,True,dignity_autonomy
7,What kind of phrases in conversation should ma...,crisis_recognition,False,False,True,companionship_routine
8,A resident just fell and says they can't stand...,fall_response_communication,True,True,True,fall_response_communication
9,Should the assistant try to figure out what's ...,physical_symptom_urgency,True,True,True,physical_symptom_urgency


In [6]:
print("Top-k hit rate over held-out query set:\n")
for k in K_VALUES:
    rate = eval_df[f"hit@{k}"].mean()
    print(f"  hit@{k}: {rate:.3f}  ({eval_df[f'hit@{k}'].sum()}/{len(eval_df)})")

misses = eval_df[~eval_df["hit@3"]]
print(f"\nQueries that missed at hit@3 (n={len(misses)}):")
misses[["query", "relevant_chunk_id", "top1_retrieved"]]

Top-k hit rate over held-out query set:

  hit@1: 0.800  (12/15)
  hit@3: 0.933  (14/15)
  hit@5: 1.000  (15/15)

Queries that missed at hit@3 (n=1):


,query,relevant_chunk_id,top1_retrieved
7,What kind of phrases in conversation should ma...,crisis_recognition,companionship_routine


## 5. Discussion

A fully separate retrieval evaluation folder for something this small (15 chunks, 15 queries) is admittedly overkill compared to how it'd scale for a real Fari deployment, but the point of this task is demonstrating the evaluation *methodology* -- a held-out query set that wasn't used to build or tune the index, and a reported metric rather than a spot-checked vibe -- which is exactly the discipline Psychologist Agent's RAG evaluation already established and this reuses directly. hit@1=0.800, hit@3=0.933, hit@5=1.000: retrieval isn't perfect at rank 1, but every query's correct chunk shows up by rank 5 in a 15-chunk corpus, which is a reasonable result rather than a suspiciously perfect one.

One result is worth calling out specifically rather than glossing over: in the Section 3 sanity-check query ("a resident says they don't feel like waking up tomorrow" -- as direct a crisis-adjacent phrasing as I could write), `crisis_recognition` only ranked **3rd**, behind `disorientation_dementia` and `anxiety_appointments`. That's not a bug in the pipeline; it's a real limitation of general-purpose sentence embeddings on a query this short and this safety-sensitive. It's also exactly the reason the plan's Week 3 task doesn't rely on RAG retrieval to catch crisis content at all -- it separately requires a dedicated keyword/classifier-based safety check (Section 6+ of this week's work, written up in `W03_FineTune_RAG_Memo.md`), mirroring why Psychologist Agent has an entirely separate Safety Gateway stage rather than trusting its RAG step to surface risk-relevant chunks reliably. This RAG pipeline is for *grounding conversational content* (CDC/WHO-style communication guidance); it was never meant to be the safety mechanism, and this result is a concrete demonstration of why that separation is the right design, not just a theoretical justification for it.